# Export WebGPU detector and optical viewers

The bundle includes a [photon camera](photon_camera.ipynb) at `camera.html`,
optical-event diagnostics at `physics.html`, and an opaque detector viewer at
`index.html`. The photon camera simulates millions of optical photons and
renders their illumination through glass, fluorescent coatings, scattering
media and a TPB-coated PMT. Controls select photon count, seed, orbit and exposure.

Use **TriChroma (GPU, pimm-bench)** on this host. Export runs on the CPU;
after export, the browser GPU performs simulation and rendering.
The main Theia detector has approximately 50,000 twenty-inch PMTs; runtime
counts come from scene metadata. PixelTPC keeps its original area-averaged
pixel surfaces. Use [detector_viewer.ipynb](detector_viewer.ipynb) for the
six analytic wire planes and [optical_showcase.ipynb](optical_showcase.ipynb)
for native GPU optical diagnostics.


In [ ]:
from pathlib import Path
import sys
root = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "chroma-lite").is_dir() and (p / "chroma-lar").is_dir()), None)
if root is None:
    raise RuntimeError("Start Jupyter from the trichroma checkout")
for source in (root / "chroma-lite", root / "chroma-lar"):
    if str(source) not in sys.path:
        sys.path.insert(0, str(source))
from chroma_lar.viewer_examples import build_viewer_example
from chroma.triton.webgpu.export import export_example, copy_browser_assets
from chroma_lar.webgpu_physics import export_physics_catalog
from chroma_lar.photon_camera import export_camera_catalog


In [ ]:
output = root / "notebooks" / "webgpu_detectors"
scenes = []
for name, source in (("theia", "theia"), ("reflect3wires", "reflect3wires-mesh"), ("pixelTPC", "pixelTPC-resolved")):
    example = build_viewer_example(source)
    scene, _ = export_example(example, output, name=name, compress=True)
    scenes.append(scene)
    print(name, scene["geometry"], f"{scene['byte_length']/1e6:.2f} MB")
copy_browser_assets(output, scenes)
camera_catalog = export_camera_catalog(output)
print("Camera scenes:", [scene["name"] for scene in camera_catalog["scenes"]])
physics_catalog = export_physics_catalog(output)
print("Optical scenes:", [scene["name"] for scene in physics_catalog["scenes"]])


Download the ZIP below, unzip it on your own computer, then run:

```bash
python -m http.server 8765 --bind 127.0.0.1 --directory webgpu_detectors
```

Open **http://localhost:8765/physics.html** for the optical laboratory or
**http://localhost:8765/index.html** for the detector explorer. Use current
Chrome/Edge with WebGPU and hardware acceleration. To serve from this remote
host instead, forward the port with `ssh -L 8765:localhost:8765 HOST` first.
The GPU used is always the one available to your browser.

The optical lab supports **1–30 million photons** per event and records up to
2,048 real trajectories from that event. Histograms count the full population.
Time gating redraws cached trajectories without rerunning physics. This is a
bounded float32 implementation of the three exported demonstrations, with
CPU-oracle validation recorded separately; it is not a generic detector backend.


In [ ]:
import shutil
from IPython.display import FileLink, display
archive = shutil.make_archive(str(output), "zip", root_dir=output.parent, base_dir=output.name)
display(FileLink(str(Path(archive).relative_to(Path.cwd()))))
print("Server command:", f"python -m http.server 8765 --bind 127.0.0.1 --directory {output}")
